# Evaluation design

In this notebook I set up how I'll test my models: a train/test split by lifter, and 5 folds on the training set for comparing methods. I don't train any models here.

In [1]:
# Add the project folder to the path so `from src import ...` works when this
# notebook is run from the notebooks folder.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.clean import clean
from src.evaluation import SEED, make_cv, split_by_lifter
from src.features import FEATURES, build_features
from src.paths import INTERIM

# The audit has already written the cohort, so I don't need to run it again here.
cohort = pd.read_csv(INTERIM / 'audit_cohort.csv.gz', dtype=str, keep_default_na=False)
cleaned, _ = clean(cohort)
table = build_features(cleaned)
print('modelling table:', table.shape)

modelling table: (461828, 14)


## 1. Split by lifter

In [2]:
train, test = split_by_lifter(table)

X_train, y_train, groups_train = train[FEATURES], train['made'], train['Name']
X_test, y_test, groups_test = test[FEATURES], test['made'], test['Name']

## 2. Check no lifter is on both sides

In [3]:
assert not set(groups_train) & set(groups_test), 'A lifter appears in both train and test'
print('No lifter appears in both train and test.')

No lifter appears in both train and test.


## 3. Size of each side

In [4]:
pd.DataFrame({
    'rows': [len(train), len(test)],
    'lifters': [groups_train.nunique(), groups_test.nunique()],
    'success_pct': [round(100 * y_train.mean(), 2), round(100 * y_test.mean(), 2)],
}, index=['train', 'test'])

,rows,lifters,success_pct
train,369880,145781,61.02
test,91948,36446,60.69


## 4. The 5 folds on the training set

In [5]:
cv = make_cv()
folds = []
for fold, (_, val_rows) in enumerate(cv.split(X_train, y_train, groups_train), 1):
    folds.append({'fold': fold, 'validation rows': len(val_rows),
                  'success_pct': round(100 * y_train.iloc[val_rows].mean(), 2)})
pd.DataFrame(folds).set_index('fold')

,validation rows,success_pct
fold,,
1,73975,61.02
2,73982,61.02
3,73976,61.02
4,73974,61.02
5,73973,61.02


## Why I set it up this way

A lot of lifters compete at more than one meet, so I split by lifter rather than by row. If the same person ended up on both sides, the model would be tested on people it had already seen and its score would look better than it really is. I've put the test set aside now and I won't touch it again until I compare my final models. Until then I'll compare methods using the 5 folds on the training set. My main score is balanced accuracy, because 61% of lifts succeed, so a model could look decent just by guessing "success" every time.